[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_30/3_stats.ipynb)

# Day 30: Statistics, A/B testing and product (hard)

**Light day.** Three short questions (about 15 minutes) to check that the core tools come back fast, then the checklist for the day before the exam. No new material today. If something here feels slow, read the matching handbook section once and stop there. Rest is part of the plan.

**How to work:** Answer in two layers: the number (code) and one plain sentence a product manager understands. Name the method and its assumptions before computing.

**Today (15 min):**

| # | Question | Level | Minutes |
|---|---|---|---|
| Q1 | Five-minute sizing | medium | 5 |
| Q2 | Rapid fire: one line each | medium | 5 |
| Q3 | The 30-second readout | medium | 5 |

**Handbook:** [Statistics, A/B testing and product](https://mahsa-agz.github.io/ds-handbook/#stats), [cheat sheets](https://mahsa-agz.github.io/ds-handbook/#cheat)

## Data

No dataset today. Everything fits in a few lines of numpy and scipy.

## Setup

Run this cell once before the questions.

In [ ]:
# Finds the data folder (local copy of the repository, or /content/data in Colab) and downloads a missing file
# from the ds-handbook repository on GitHub (or from its original source).
import os, sys, urllib.request

RAW = "https://raw.githubusercontent.com/mahsa-agz/mahsa-agz.github.io/main/ds-handbook/exam_prep/data/"

def _find_data():
    if "google.colab" in sys.modules:
        os.makedirs("/content/data", exist_ok=True)
        return "/content/data"
    for p in ("../data", "data", "exam_prep/data"):
        if os.path.isdir(p):
            return p
    os.makedirs("data", exist_ok=True)
    return "data"

DATA = _find_data()
print("data folder:", DATA)

def data_path(name, url=None):
    """Local path of a data file; downloads it from the repository (or from url) if it is not there yet."""
    path = os.path.join(DATA, name)
    if not os.path.exists(path):
        for source in (RAW + name, url):
            if not source:
                continue
            try:
                print("downloading", name)
                urllib.request.urlretrieve(source, path)
                break
            except Exception as e:
                print("  could not download from", source, "->", e)
    return path

import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf
pd.set_option("display.width", 120)

from scipy.stats import norm

---
### Q1. Five-minute sizing

*medium, about 5 min*

Checkout conversion is 10%. The PM wants to detect a **5% relative** lift (to 10.5%), two-sided alpha 0.05, power 80%. 20,000 eligible users arrive per day.

1. Users per group? Check your answer with the rule of thumb `16 p(1-p) / delta^2`.
2. How many days should the test run?
3. If the PM can only give 20,000 users per group, what lift can the test detect?

**Say it to a PM:** explain the duration in one sentence.

In [ ]:
# your code here

<details><summary><b>Hint 1</b></summary>

Signal: 'how many users / how long'. Method: power and MDE for two proportions.

</details>

<details><summary><b>Hint 2</b></summary>

1. `n = (z_0.975 + z_0.8)^2 (p1 q1 + p2 q2) / (p2 - p1)^2`. 2. Days = `2n / daily users`, rounded up to whole weeks. 3. Invert: `MDE = (z_0.975 + z_0.8) sqrt(2 p q / n)`.

</details>

<details><summary><b>Solution</b></summary>

```python
p1, p2 = 0.10, 0.105                        # baseline 10%, MDE 5% relative
za, zb = norm.ppf(0.975), norm.ppf(0.80)
n = (za + zb) ** 2 * (p1 * (1 - p1) + p2 * (1 - p2)) / (p2 - p1) ** 2
print(f"exact-ish: {n:,.0f} per group; rule of thumb 16 p(1-p) / delta^2: {16 * p1 * (1 - p1) / (p2 - p1) ** 2:,.0f}")
print(f"with 20,000 eligible users a day split 50/50: {2 * n / 20_000:.1f} days -> run 7 days (a full week)")
print(f"MDE if we only get 20,000 users per group: {(za + zb) * np.sqrt(2 * p1 * (1 - p1) / 20_000) / p1:.1%} relative")
# exact-ish: 57,760 per group; rule of thumb 16 p(1-p) / delta^2: 57,600
# with 20,000 eligible users a day split 50/50: 5.8 days -> run 7 days (a full week)
# MDE if we only get 20,000 users per group: 8.4% relative
```

**Why:** About 57.8k users per group (the rule of thumb gives 57.6k because `(1.96 + 0.84)^2` is about 7.85, so `2 x 7.85` is about 16). That is about 6 days of traffic, but tests should cover whole weeks to include weekday and weekend behaviour, so run 7 days (14 if novelty is a concern). With only 20k per group the smallest detectable lift is about 8.4% relative.

**Say it to a PM:** "To reliably see a 5% lift we need about 58,000 users per group, which is 6 days of traffic, so we run one full week to cover weekends."

**Common mistakes:** Using the relative MDE (0.05) as delta. Stopping on day 6 in the middle of a week. Forgetting that the variance uses p near 0.10, not 0.5.

**Learn more:** [stats-power-mde](https://mahsa-agz.github.io/ds-handbook/#stats-power-mde), [stats-ab-design](https://mahsa-agz.github.io/ds-handbook/#stats-ab-design)

</details>

---
### Q2. Rapid fire: one line each

*medium, about 5 min*

Answer each in one sentence, out loud, in under 20 seconds:

1. The first thing you check when a metric drops.
2. What a p-value is (and is not).
3. When you randomize clusters instead of users.
4. Why peeking at a fixed-horizon test is a problem.
5. The condition a CUPED covariate must meet.
6. What Simpson's paradox is, in one example.
7. How you tell a novelty effect from a learning effect.

**Say it to a PM:** item 2, in words a PM would use.

*Your answer (double-click to write):*

<details><summary><b>Hint 1</b></summary>

Signal: recall. Method: one definition plus one consequence per item.

</details>

<details><summary><b>Hint 2</b></summary>

Data quality first; P(data this extreme | no effect); interference; repeated tests inflate false wins; measured before treatment; the mix flips the direction; new users and the time trend.

</details>

<details><summary><b>Solution</b></summary>

1. **Metric drop:** is the data right (pipeline, logging, definition change), then is it unusual
compared with normal variation and the calendar, before hunting causes.
2. **p-value:** the probability of data at least this extreme if there were no effect; it is **not** the
probability that the effect is real or that the null is true.
3. **Cluster randomization:** when units interfere (friends, shared supply or budget), so you randomize groups that
contain the interaction, and analyze at the cluster level.
4. **Peeking:** every look is another chance for noise to cross the threshold; daily looks with 1.96 can push false
wins from 5% to about 25%, so use a sequential design.
5. **CUPED:** the covariate must be measured before randomization (unaffected by treatment) and correlated with the
metric; the variance falls by about `rho^2`.
6. **Simpson's paradox:** a trend in every subgroup reverses in the total because the groups have different mixes;
for example phone customers churn more overall but less inside DSL, because all Fiber customers have phone.
7. **Novelty vs learning:** novelty decays over time and is absent for new users; learning grows over time. Plot the
lift by day since first exposure, and compare new and existing users.

**Say it to a PM:** "The p-value tells us how surprising this result would be if the change did nothing; a small
value means 'unlikely to be pure luck', not 'a 95% chance that it works'."

**Why:** Fast, exact definitions are the foundation examiners test before case questions; each answer here has a definition and a consequence.

**Learn more:** [stats-hypothesis-tests](https://mahsa-agz.github.io/ds-handbook/#stats-hypothesis-tests), [stats-ab-pitfalls](https://mahsa-agz.github.io/ds-handbook/#stats-ab-pitfalls), [stats-metric-drop](https://mahsa-agz.github.io/ds-handbook/#stats-metric-drop)

</details>

---
### Q3. The 30-second readout

*medium, about 5 min*

You get this readout of a 2-week test of a new comment ranking (user-level randomization, SRM check passed):

| Metric | Change | 95% CI |
|---|---|---|
| Comments per user (primary) | +3.1% | +1.9% to +4.3% |
| Watch time per user | +0.1% | -0.3% to +0.5% |
| Reports per 1k comments (guardrail) | -6.0% | -11.0% to -1.0% |
| Day-7 retention | +0.2% | -0.4% to +0.8% |

Write the 30-second summary for the PM: result, confidence, risks, decision.

**Say it to a PM:** this whole question is the PM sentence; keep it under 60 words.

*Your answer (double-click to write):*

<details><summary><b>Hint 1</b></summary>

Signal: a clean readout. Method: lead with the decision, then the main number with its range, then guardrails and one risk.

</details>

<details><summary><b>Hint 2</b></summary>

1. Primary up and clearly positive. 2. Guardrail improved. 3. Watch time and retention flat. 4. Risk: novelty over 2 weeks. 5. Decision: launch with a holdout.

</details>

<details><summary><b>Solution</b></summary>

**Model answer (about 55 words):** "I recommend launching. Comments per user rose 3.1%, and we are
confident it is between 2% and 4%. Reports per thousand comments fell 6%, so the extra comments are not toxic.
Watch time and retention did not change. The main risk is novelty after only two weeks, so we keep a 5% holdout
for a month to confirm."

**Why it works:** decision first, one number with its range in plain words, the guardrail that answers the obvious
worry, the neutral metrics in one sentence, and a concrete plan for the remaining risk.

**Why:** Examiners often end a case with 'summarize for the PM'. A short, ordered summary with numbers and a decision shows judgement more than any formula.

**Learn more:** [stats-launch-decisions](https://mahsa-agz.github.io/ds-handbook/#stats-launch-decisions), [stats-case-framework](https://mahsa-agz.github.io/ds-handbook/#stats-case-framework)

</details>

---
## Checklist for the day before the exam

**Morning (60 to 90 minutes, then stop)**
- Redo only the open items in your mistake log, one pass, no new topics.
- Read the cheat sheets once: [stats tests](https://mahsa-agz.github.io/ds-handbook/#cheat-stats-tests), [A/B testing](https://mahsa-agz.github.io/ds-handbook/#cheat-ab-testing), [exam day](https://mahsa-agz.github.io/ds-handbook/#cheat-exam-day).
- Say the case framework out loud once: clarify, goal, metrics (primary, drivers, guardrails), design, analysis, decision.
- Say three formulas out loud: `SE = sqrt(p(1-p)/n)`, `n = 16 sd^2 / delta^2` per group, CUPED variance times `1 - rho^2`.

**Stories and questions**
- Two project stories in 90 seconds each: problem, what you did, the number that changed, what you learned.
- One story about a wrong analysis you caught (or made) and how you fixed it.
- Three questions for the examiner about how the team uses experiments and metrics.

**Logistics**
- Check the time zone, the link, camera, microphone and a quiet room; test the coding pad if there is one.
- Prepare paper and pen for formulas and a calculator you are allowed to use.
- Water, a light meal before, and a fixed stop time tonight.

**In the exam**
- Repeat the question in your own words and ask one or two clarifying questions before you compute.
- Name the method and its assumption before the number; give the number with its range; end with a decision.
- If you get stuck, say what you would check next. A clear partial plan scores better than silence.
- Every answer ends with the sentence you would say to a PM.

**Evening:** no studying after dinner. Sleep is worth more than one more topic.

---
## Redo list

Write down every question you got wrong, solved only after Hint 2, or finished over time. Solve it again from scratch 3 days and 7 days later.

**Next:** [AI and ML](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_30/4_ai.ipynb)